This notebook contains a test for the function `toNFA` implemented in the notebook `03-Regexp-2-NFA.ipynb`.
In order to write regular expressions conveniently, we need the parser implemented in `RegExp-Parser.ipynb`.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "AutomataTypes.fsx"
#load "RegExpParser.fsx"
#load "RegExpToNfa.fsx"
#load "NfaToDfa.fsx"
#load "FsmToDot.fsx"

In [ ]:
open AutomataTypes
open RegExpParser
open RegExpToNfa
open NfaToDfa
open FsmToDot

The example `a*ba*` recognizes exactly the words with one `b` and any number of `a`s before and after it:
$$a^*\cdot b\cdot a^*.$$
The Python notebook's displayed formula described a different language; this explanation follows its actual executable example.

In [ ]:
let r = parse "a*ba*"
r

We use `regexp2nfa` to create an <span style="font-variant:small-caps;">NFA</span>  that accepts the language 
described by the regular expression `r`.

In [ ]:
let nfa = regexp2nfa (set ['a';'b']) r
printfn "%s" (nfa2string nfa)
let nfaDot = nfa2dot nfa
nfaDot

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderSvg nfaDot), "text/html") |> ignore

After having constructed a non-deterministic <span style="font-variant:small-caps;">Fsm</span> for the regular expression,
we convert it into a deterministic <span style="font-variant:small-caps;">Fsm</span>.

In [ ]:
let dfa = nfa2dfa nfa
let dfaDot, stateNames = dfa2dot dfa
dfaDot

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderSvg dfaDot), "text/html") |> ignore

Note that the resulting DFA is not minimal, since we can identify some states.

In [ ]:
stateNames

In [ ]:
for word, expected in ["b", true; "aabaa", true; "baaa", true; "", false; "aaa", false; "bb", false] do
    checkNotebook (acceptsNfa nfa word = expected)
    checkNotebook (acceptsDfa dfa word = expected)
checkNotebook (regexp2nfa (set ['a';'b']) r = nfa)
for expression, emptyAccepted in [Empty, false; Epsilon, true; Star Empty, true] do
    let fragment = regexp2nfa (set ['a';'b']) expression
    checkNotebook (acceptsNfa fragment "" = emptyAccepted)